# S13 — Permutation importance sobre predictores originales

Versión notebook auto. Ejecuta de arriba abajo.

**Objetivo:** validar que los modelos congelados reproducen las métricas oficiales del held-out test y, solo entonces, calcular permutation importance permutando cada predictor original antes de aplicar `scaler → PCA → modelo`.

**Revisa antes de ejecutar:** la variable `BASE_DIR` debe apuntar a tu carpeta `Version_Final`.


**v4:** detección robusta para evitar seleccionar un objeto PCA como modelo.

In [1]:
# ======================================================================================
# S13 — Permutation importance sobre predictores originales — versión AUTO

In [2]:
# ======================================================================================
# Qué hace:
# 1) Usa artefactos finales congelados: model/pipeline, scaler, PCA, means, feature_names.
# 2) Reproduce el preprocesado correcto: MinMaxScaler -> mean-centering -> PCA -> componentes retenidos.
# 3) Valida que el baseline reproduce las métricas oficiales del held-out test.
# 4) Si la validación pasa, calcula permutation importance permutando predictores originales.
#
# Uso:
# - Pon BASE_DIR a tu carpeta Version_Final.
# - Ejecuta de arriba abajo.
# - Si no encuentra algún modelo, rellena MANUAL_ARTIFACTS con las rutas que imprime.

In [3]:
# ======================================================================================

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    import joblib
except Exception as e:
    raise ImportError("Instala joblib: pip install joblib") from e

try:
    import tensorflow as tf
except Exception:
    tf = None

from sklearn.metrics import (
    recall_score, precision_score, f1_score, accuracy_score,
    balanced_accuracy_score, roc_auc_score, average_precision_score,
    confusion_matrix
)

In [4]:
# ======================================================================================
# CONFIGURACIÓN CENTRAL

In [55]:
# ======================================================================================

BASE_DIR = Path("/Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final")
OUT_DIR = BASE_DIR / "final_permutation_importance_original_predictors_S13"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ALIGNED_FILES = {
    "victimization": BASE_DIR / "aligned_test_exports" / "victimization_aligned_test_rowlevel.csv",
    "perpetration": BASE_DIR / "aligned_test_exports" / "perpetration_aligned_test_rowlevel.csv",
    "overlap": BASE_DIR / "aligned_test_exports" / "overlap_aligned_test_rowlevel.csv",
}

THRESHOLDS = {
    "victimization": 0.500,
    "perpetration": 0.500,
    "overlap": 0.523164,
}

OFFICIAL = {
    "victimization": {
        "recall_sensitivity": 0.871,
        "specificity": 0.310,
        "precision_ppv": 0.552,
        "balanced_accuracy": 0.591,
    },
    "perpetration": {
        "recall_sensitivity": 0.914,
        "specificity": 0.301,
        "precision_ppv": 0.286,
        "balanced_accuracy": 0.607,
    },
    "overlap": {
        "recall_sensitivity": 0.809,
        "specificity": 0.535,
        "precision_ppv": 0.289,
        "balanced_accuracy": 0.672,
    },
}

VALIDATION_TOLERANCE = 0.015
N_REPEATS = 30
RANDOM_STATE = 123

# ----------------------------------------------------------------------
# NUEVO EN v3:
# Los aligned_test_rowlevel.csv NO contienen las 27 columnas originales.
# Este notebook busca automáticamente una matriz analítica de features con 3767 filas
# y usa idx_original como índice de la muestra analítica, NO como índice bruto de 4024.
# ----------------------------------------------------------------------

# Puedes dejarlo en None. El script intentará encontrar automáticamente:
#   df_victima_feat.csv, df_perpetrador_feat.csv, lista_global_vars.csv, etc.
# Si quieres forzarlo manualmente, pon rutas absolutas aquí.
SOURCE_FEATURE_MATRIX_PATHS = {
    "victimization": "/Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_victim/df_victima_feat.csv",

    "perpetration": "/Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_perpetrator/content/perpetrator_v3/df_perpetrador_feat.csv",

    "overlap": "/Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_perpetrator/content/perpetrator_v3/df_perpetrador_feat.csv",
}

# idx_original de los aligned files se interpreta como índice posicional dentro de la matriz analítica.
SOURCE_ROW_ID_COL = None

# Artefactos oficiales congelados exportados desde los notebooks finales.
# IMPORTANTE: means.csv es necesario porque el PCA se entrenó con MinMaxScaler + mean-centering.
ARTIFACT_BASE = BASE_DIR / "final_model_artifacts_for_S13"

MANUAL_ARTIFACTS = {
    "victimization": {
        "pipeline": None,
        "scaler": ARTIFACT_BASE / "victimization" / "scaler.joblib",
        "pca": ARTIFACT_BASE / "victimization" / "pca.joblib",
        "means": ARTIFACT_BASE / "victimization" / "means.csv",
        "model": ARTIFACT_BASE / "victimization" / "model.joblib",
        "feature_names": ARTIFACT_BASE / "victimization" / "feature_names.csv",
    },
    "perpetration": {
        "pipeline": None,
        "scaler": ARTIFACT_BASE / "perpetration" / "scaler.joblib",
        "pca": ARTIFACT_BASE / "perpetration" / "pca.joblib",
        "means": ARTIFACT_BASE / "perpetration" / "means.csv",
        "model": ARTIFACT_BASE / "perpetration" / "model.keras",
        "feature_names": ARTIFACT_BASE / "perpetration" / "feature_names.csv",
    },
    "overlap": {
        "pipeline": None,
        "scaler": ARTIFACT_BASE / "overlap" / "scaler.joblib",
        "pca": ARTIFACT_BASE / "overlap" / "pca.joblib",
        "means": ARTIFACT_BASE / "overlap" / "means.csv",
        "model": ARTIFACT_BASE / "overlap" / "model.joblib",
        "feature_names": ARTIFACT_BASE / "overlap" / "feature_names.csv",
    },
}

DEFAULT_FEATURES = [
    "PAÍS", "ETNIA.BN", "EDAD",
    "FUGAS.BN", "ABUSOSUBS1", "ABUSOSUBS2",
    "CONVIVEN.1", "CONVIVEN.2", "CONVIVEN.3", "CONVIVEN.4", "CONVIVEN_H", "CONVIVEN_0",
    "AUTOEFIC.MEAN", "AUTOEFIC.VAR",
    "IMPULS.MEAN", "IMPULS.MEDIAN", "IMPULS.VAR",
    "APOYO.MEAN", "APOYO.MEDIAN", "APOYO.VAR",
    "MORAL.MEAN", "MORAL.VAR",
    "PORNO.T",
    "GENERO.BN0", "GENERO.BN1",
    "ORIENTSEX.BN0", "ORIENTSEX.BN1",
]


In [30]:
# ======================================================================================
# HELPERS

In [57]:
# ======================================================================================

def norm(x):
    return str(x).strip().lower().replace(" ", "_").replace("-", "_").replace("í", "i")

def exists_or_none(x):
    if x is None or str(x).strip() == "":
        return None
    p = Path(x)
    return p if p.exists() else None

def find_col(df, candidates, required=True, kind="column"):
    norm_to_original = {norm(c): c for c in df.columns}
    for cand in candidates:
        if norm(cand) in norm_to_original:
            return norm_to_original[norm(cand)]
    for c in df.columns:
        cn = norm(c)
        if any(norm(cand) in cn for cand in candidates):
            return c
    if required:
        raise ValueError(f"No puedo detectar {kind}. Columnas disponibles: {list(df.columns)}")
    return None

def load_obj(path):
    if path is None:
        return None
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    suffix = path.suffix.lower()
    if suffix in [".joblib", ".pkl", ".pickle"]:
        return joblib.load(path)
    if suffix in [".keras", ".h5"]:
        if tf is None:
            raise ImportError("TensorFlow no disponible, pero el modelo es Keras/H5.")
        return tf.keras.models.load_model(path, compile=False)
    raise ValueError(f"Tipo de artefacto no soportado: {path}")

def read_feature_names(path):
    """Lee feature_names.csv robustamente.

    Importante: nuestros feature_names.csv se exportaron con header=False.
    Si se leen con pd.read_csv(path) sin header=None, pandas interpreta la primera
    feature como cabecera y se pierde una columna (27 -> 26).
    """
    if path is None:
        return None
    path = Path(path)
    if not path.exists():
        return None

    # Primero intenta formato con cabecera explícita.
    df_head = pd.read_csv(path)
    for c in ["feature", "feature_name", "name", "predictor", "variable"]:
        if c in df_head.columns:
            vals = df_head[c].dropna().astype(str).tolist()
            if vals:
                return vals

    # Formato real usado aquí: una columna sin header.
    df_raw = pd.read_csv(path, header=None)
    vals = df_raw.iloc[:, 0].dropna().astype(str).tolist()

    # Por si accidentalmente se guardó con un header genérico en la primera fila.
    bad_headers = {"feature", "feature_name", "name", "predictor", "variable", "0"}
    if vals and vals[0].strip().lower() in bad_headers:
        vals = vals[1:]

    return vals

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fp) if (tn + fp) else np.nan

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fn) if (tn + fn) else np.nan

def compute_metrics(y_true, score, threshold):
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)
    y_pred = (score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    out = {
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "recall_sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": specificity_score(y_true, y_pred),
        "precision_ppv": precision_score(y_true, y_pred, zero_division=0),
        "npv": npv_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1_positive": f1_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
    }
    try:
        out["roc_auc"] = roc_auc_score(y_true, score)
    except Exception:
        out["roc_auc"] = np.nan
    try:
        out["pr_auc_average_precision"] = average_precision_score(y_true, score)
    except Exception:
        out["pr_auc_average_precision"] = np.nan
    return out

In [48]:
# ======================================================================================
# AUTO-DISCOVERY DE ARTEFACTOS

In [58]:
# ======================================================================================

# v4: detección robusta de artefactos.
# Problema corregido: en v3 algunos archivos PCA podían ser seleccionados por error como "model".
# Ahora el script:
# - excluye pca/scaler/features/predictions/metrics como candidatos a modelo;
# - carga los candidatos y valida que tengan predict_proba() o predict();
# - prioriza nombres/directorios coherentes con cada outcome y con modelos finales.

BAD_MODEL_WORDS = [
    "pca", "scaler", "minmax", "feature", "features", "loading", "variance",
    "aligned", "rowlevel", "prediction", "predictions", "metrics", "summary",
    "bootstrap", "calibration", "threshold", "subgroup", "audit", "candidate",
    "comparison", "table", "csv"
]

GOOD_MODEL_WORDS = {
    "victimization": ["tree", "decision", "pruned", "arbol", "dt", "classifier", "clf", "model", "modelo", "best", "selected", "final"],
    "perpetration": ["dnn", "neural", "nn", "keras", "h5", "model", "modelo", "best", "selected", "final"],
    "overlap": ["logistic", "logreg", "lr", "classifier", "clf", "model", "modelo", "best", "selected", "final", "sw_pos1"],
}


def discover_all_files(base_dir):
    patterns = ["*.joblib", "*.pkl", "*.pickle", "*.keras", "*.h5", "*.csv", "*.json"]
    rows = []
    for pat in patterns:
        for p in base_dir.rglob(pat):
            rows.append({
                "path": str(p),
                "name": p.name,
                "parent": str(p.parent),
                "suffix": p.suffix.lower(),
                "name_l": p.name.lower(),
                "parent_l": str(p.parent).lower(),
            })
    return pd.DataFrame(rows).drop_duplicates("path") if rows else pd.DataFrame(columns=["path", "name", "parent", "suffix", "name_l", "parent_l"])


def outcome_tokens(outcome):
    if outcome == "victimization":
        return ["victim", "victimiz", "victima", "victimization", "victimizacion", "final_victim"]
    if outcome == "perpetration":
        return ["perp", "perpetr", "perpetration", "perpetrator", "final_perp"]
    if outcome == "overlap":
        return ["overlap", "intersect", "intersection", "common", "sw_pos1", "final_overlap"]
    return [outcome]


def score_path_for_outcome(row, outcome):
    text = (row["name_l"] + " " + row["parent_l"]).lower()
    score = 0
    for tok in outcome_tokens(outcome):
        if tok in text:
            score += 10
    for tok in ["final", "trainonly", "pca_trainonly", "version_final", "selected", "best"]:
        if tok in text:
            score += 2
    for tok in GOOD_MODEL_WORDS.get(outcome, []):
        if tok in text:
            score += 3
    return score


def is_usable_model_path(path):
    """Carga un candidato y comprueba que es realmente un modelo predictor."""
    try:
        obj = load_obj(path)
    except Exception:
        return False, None
    if obj is None:
        return False, None
    # PCA tiene transform pero no predict; scaler tampoco. Necesitamos predict o predict_proba.
    if hasattr(obj, "predict_proba") or hasattr(obj, "predict"):
        return True, obj.__class__.__name__
    return False, obj.__class__.__name__


def pick_best(df, outcome, kind):
    if df.empty:
        return None
    d = df.copy()
    text = (d["name_l"] + " " + d["parent_l"]).str.lower()

    if kind == "pipeline":
        mask = text.str.contains("pipeline", regex=False) & d["suffix"].isin([".joblib", ".pkl", ".pickle"])
    elif kind == "scaler":
        mask = text.str.contains("scaler|minmax", regex=True) & d["suffix"].isin([".joblib", ".pkl", ".pickle"])
    elif kind == "pca":
        # Preferimos objetos PCA explícitos, no cualquier archivo cuyo directorio tenga pca_trainonly.
        mask = d["name_l"].str.contains("pca", regex=False) & d["suffix"].isin([".joblib", ".pkl", ".pickle"])
    elif kind == "feature_names":
        mask = text.str.contains("feature|features|pca_input", regex=True) & d["suffix"].eq(".csv")
    elif kind == "model":
        allowed_suffix = [".joblib", ".pkl", ".pickle", ".keras", ".h5"] if outcome == "perpetration" else [".joblib", ".pkl", ".pickle"]
        mask = d["suffix"].isin(allowed_suffix)
        # Excluir artefactos que NO son modelos aunque estén en carpeta final/pca_trainonly.
        bad_regex = "|".join(BAD_MODEL_WORDS)
        mask = mask & (~text.str.contains(bad_regex, regex=True, na=False))
        # Incluir solo rutas con palabras plausibles de modelo.
        good_regex = "|".join(GOOD_MODEL_WORDS.get(outcome, []))
        mask = mask & text.str.contains(good_regex, regex=True, na=False)
    else:
        return None

    cand = d[mask].copy()
    if cand.empty:
        return None

    cand["score"] = cand.apply(lambda r: score_path_for_outcome(r, outcome), axis=1)
    if kind in ["model", "pipeline"]:
        cand = cand[cand["score"] >= 10].copy()
        if cand.empty:
            return None

    cand = cand.sort_values(["score", "path"], ascending=[False, True]).reset_index(drop=True)

    # Para modelos/pipelines, validar que el objeto cargado sea realmente predictivo.
    if kind in ["model", "pipeline"]:
        checked_rows = []
        for _, r in cand.iterrows():
            path = Path(r["path"])
            ok, clsname = is_usable_model_path(path)
            checked_rows.append({"path": str(path), "score": r["score"], "usable_model": ok, "class": clsname})
            if ok:
                return path
        checked = pd.DataFrame(checked_rows)
        checked.to_csv(OUT_DIR / f"S13_checked_{outcome}_{kind}_candidates.csv", index=False)
        return None

    return Path(cand.iloc[0]["path"])


def build_auto_artifacts(base_dir):
    df = discover_all_files(base_dir)
    interesting = df[
        (df["name_l"] + " " + df["parent_l"]).str.contains(
            "model|modelo|tree|pruned|clf|logistic|logreg|neural|dnn|keras|h5|scaler|minmax|pca|feature|pca_input|pipeline|aligned|rowlevel",
            regex=True,
            na=False,
        )
    ].copy()
    auto = {}
    for outcome in ["victimization", "perpetration", "overlap"]:
        auto[outcome] = {
            "pipeline": pick_best(df, outcome, "pipeline"),
            "scaler": pick_best(df, outcome, "scaler"),
            "pca": pick_best(df, outcome, "pca"),
            "means": None,
            "model": pick_best(df, outcome, "model"),
            "feature_names": pick_best(df, outcome, "feature_names"),
        }
    return auto, interesting


def combine_artifacts(manual, auto):
    out = {}
    for outcome in ["victimization", "perpetration", "overlap"]:
        out[outcome] = {}
        for k in ["pipeline", "scaler", "pca", "means", "model", "feature_names"]:
            m = exists_or_none(manual[outcome].get(k))
            out[outcome][k] = m if m is not None else auto[outcome].get(k)
    return out


AUTO_ARTIFACTS, artifact_candidates = build_auto_artifacts(BASE_DIR)
ARTIFACTS = combine_artifacts(MANUAL_ARTIFACTS, AUTO_ARTIFACTS)

print("\nCANDIDATOS DE ARTEFACTOS ENCONTRADOS")
print("=" * 90)
if artifact_candidates.empty:
    print("No he encontrado candidatos en BASE_DIR. Revisa BASE_DIR.")
else:
    candidates_out = OUT_DIR / "S13_artifact_candidates.csv"
    artifact_candidates[["name", "parent", "suffix", "path"]].sort_values(["parent", "name"]).to_csv(candidates_out, index=False)
    print("Guardado listado de candidatos en:", candidates_out)
    print(artifact_candidates[["name", "parent", "suffix", "path"]].sort_values(["parent", "name"]).head(120).to_string(index=False))

print("\nARTIFACTS QUE USARÁ EL SCRIPT")
print("=" * 90)
missing_any = False
for outcome, cfg in ARTIFACTS.items():
    print("\n", outcome)
    for k, v in cfg.items():
        ok = v is not None and Path(v).exists()
        print(f"  {k:14s} -> {ok} | {v}")
    if cfg.get("pipeline") is None and cfg.get("model") is None:
        missing_any = True
        print("  >>> FALTA MODELO/PIPELINE")

if missing_any:
    print("\nFalta algún modelo/pipeline. Rellena MANUAL_ARTIFACTS con rutas exactas y vuelve a ejecutar.")
    raise SystemExit(1)



CANDIDATOS DE ARTEFACTOS ENCONTRADOS
Guardado listado de candidatos en: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_permutation_importance_original_predictors_S13/S13_artifact_candidates.csv
                                               name                                                                                                                                                                                                  parent  suffix                                                                                                                                                                                                                               path
                  overlap_aligned_test_rowlevel.csv                                                                                             /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/aligned_test_exports    .csv             

In [34]:
# ======================================================================================
# PREDICTOR CONGELADO + DATOS TEST

In [59]:
# ======================================================================================

class FrozenPredictor:
    def __init__(self, outcome, artifacts):
        self.outcome = outcome
        self.artifacts = artifacts
        self.pipeline = load_obj(artifacts.get("pipeline"))
        self.scaler = load_obj(artifacts.get("scaler"))
        self.pca = load_obj(artifacts.get("pca"))
        self.model_path = artifacts.get("model")
        self.model = load_obj(self.model_path)
        self.feature_names = read_feature_names(artifacts.get("feature_names"))
        self.means = None
        means_path = artifacts.get("means")
        if means_path is not None and Path(means_path).exists():
            means_df = pd.read_csv(means_path, index_col=0)
            self.means = means_df.iloc[:, 0].to_numpy(dtype=float)
        elif self.pca is not None:
            raise FileNotFoundError(
                f"{outcome}: falta means.csv. El PCA final se entrenó con MinMaxScaler + mean-centering; "
                f"exporta means.csv y actualiza MANUAL_ARTIFACTS. Ruta esperada: {means_path}"
            )

        if self.pipeline is None and self.model is None:
            raise ValueError(f"{outcome}: no hay model/pipeline.")
        if self.pipeline is None and self.model is not None:
            if not (hasattr(self.model, "predict_proba") or hasattr(self.model, "predict")):
                raise ValueError(
                    f"{outcome}: el artefacto seleccionado como model NO es un modelo predictivo: "
                    f"{self.model_path} | clase={self.model.__class__.__name__}. "
                    "Revisa MANUAL_ARTIFACTS o S13_artifact_candidates.csv."
                )

    def _model_input_dim(self):
        """Devuelve el número de columnas que espera el modelo final.
        Esto es importante porque algunos notebooks guardan un PCA completo de 27 componentes,
        pero el modelo final usa solo los primeros 18/22 componentes retenidos.
        """
        if self.model is None:
            return None

        # scikit-learn models
        n = getattr(self.model, "n_features_in_", None)
        if n is not None:
            return int(n)

        # Keras / TensorFlow models
        shape = getattr(self.model, "input_shape", None)
        if shape is not None:
            # Ej.: (None, 22) o [(None, 22)]
            if isinstance(shape, list):
                shape = shape[0]
            if len(shape) >= 2 and shape[-1] is not None:
                return int(shape[-1])

        return None

    def _transform_X(self, X_df):
        X = X_df.copy()

        if self.feature_names is not None:
            missing = [f for f in self.feature_names if f not in X.columns]
            if missing:
                raise ValueError(f"{self.outcome}: faltan features en X: {missing}")
            X = X[self.feature_names]

        if self.scaler is not None:
            X = self.scaler.transform(X)

        # PCA train-only: MinMaxScaler -> mean-centering -> PCA.
        # Sin este centrado, la baseline no reproduce Table 1.
        if self.means is not None:
            if len(self.means) != X.shape[1]:
                raise ValueError(
                    f"{self.outcome}: means.csv tiene {len(self.means)} valores, "
                    f"pero X tiene {X.shape[1]} columnas tras scaler."
                )
            X = X - self.means

        if self.pca is not None:
            X = self.pca.transform(X)

            expected = self._model_input_dim()
            if expected is not None and X.shape[1] != expected:
                if X.shape[1] > expected:
                    # PCA completo guardado; usamos los primeros componentes retenidos.
                    X = X[:, :expected]
                else:
                    raise ValueError(
                        f"{self.outcome}: el PCA transformó a {X.shape[1]} componentes, "
                        f"pero el modelo espera {expected}."
                    )

        return X

    def predict_score(self, X_df):
        if self.pipeline is not None:
            obj = self.pipeline
            if hasattr(obj, "predict_proba"):
                return obj.predict_proba(X_df)[:, 1]
            return np.asarray(obj.predict(X_df)).ravel().astype(float)

        X = self._transform_X(X_df)

        if hasattr(self.model, "predict_proba"):
            return self.model.predict_proba(X)[:, 1]

        return np.asarray(self.model.predict(X, verbose=0)).ravel().astype(float)

# ----------------------------------------------------------------------
# Alias de nombres entre versiones de notebooks.
# El modelo puede esperar GENERO.BN0, pero algunos CSVs guardan GENERO_BIN_0, etc.
# ----------------------------------------------------------------------
FEATURE_ALIASES = {
    "PAIS": ["PAÍS"],
    "PAÍS": ["PAIS"],
    "CONVIVEN_H": ["CONVIVEN.5", "CONVIVEN_H"],
    "CONVIVEN_0": ["CONVIVEN.6", "CONVIVEN_0"],
    "CONVIVEN.5": ["CONVIVEN_H", "CONVIVEN.5"],
    "CONVIVEN.6": ["CONVIVEN_0", "CONVIVEN.6"],
    "GENERO.BN0": ["GENERO_BIN_0", "GENERO.BN0"],
    "GENERO.BN1": ["GENERO_BIN_1", "GENERO.BN1"],
    "GENERO.BN2": ["GENERO_BIN_2", "GENERO.BN2"],
    "GENERO_BIN_0": ["GENERO.BN0", "GENERO_BIN_0"],
    "GENERO_BIN_1": ["GENERO.BN1", "GENERO_BIN_1"],
    "GENERO_BIN_2": ["GENERO.BN2", "GENERO_BIN_2"],
    "ORIENTSEX.BN0": ["ORIENTSEX.BN_1", "ORIENTSEX.BN0"],
    "ORIENTSEX.BN1": ["ORIENTSEX.BN_2", "ORIENTSEX.BN1"],
    "ORIENTSEX.BN2": ["ORIENTSEX.BN_3", "ORIENTSEX.BN2"],
    "ORIENTSEX.BN_1": ["ORIENTSEX.BN0", "ORIENTSEX.BN_1"],
    "ORIENTSEX.BN_2": ["ORIENTSEX.BN1", "ORIENTSEX.BN_2"],
    "ORIENTSEX.BN_3": ["ORIENTSEX.BN2", "ORIENTSEX.BN_3"],
}

def ensure_alias_columns(df):
    """Crea columnas con los nombres esperados por los modelos a partir de alias conocidos."""
    out = df.copy()
    cols = set(out.columns)
    for target, aliases in FEATURE_ALIASES.items():
        if target in cols:
            continue
        for a in aliases:
            if a in cols:
                out[target] = out[a]
                cols.add(target)
                break
    # Quita columna de índice accidental si aparece.
    # No la borramos si algún modelo absurdamente la esperase, pero no debería.
    return out

def resolve_features(available_cols, configured_features=None):
    available = list(available_cols)
    norm_map = {norm(c): c for c in available}
    if configured_features:
        features, missing = [], []
        for f in configured_features:
            if f in available:
                features.append(f)
            elif norm(f) in norm_map:
                features.append(norm_map[norm(f)])
            else:
                missing.append(f)
        return features, missing
    features = []
    for f in DEFAULT_FEATURES:
        if f in available:
            features.append(f)
        elif norm(f) in norm_map:
            features.append(norm_map[norm(f)])
    return list(dict.fromkeys(features)), []

def make_feature_groups(features):
    fset = set(features)
    groups, used = [], set()
    def add_group(name, members):
        members2 = [m for m in members if m in fset]
        if len(members2) >= 2:
            groups.append({"group": name, "features": members2})
            used.update(members2)
    add_group("Gender dummy contrast", ["GENERO.BN0", "GENERO.BN1"])
    add_group("Sexual-orientation dummy contrast", ["ORIENTSEX.BN0", "ORIENTSEX.BN1"])
    for f in features:
        if f not in used:
            groups.append({"group": f, "features": [f]})
            used.add(f)
    return groups

def candidate_search_roots(base_dir):
    roots = []
    p = Path(base_dir).resolve()
    roots.append(p)
    roots.extend(list(p.parents)[:4])
    # También prueba la carpeta de trabajo actual.
    roots.append(Path.cwd())
    # Quita duplicados manteniendo orden.
    seen, out = set(), []
    for r in roots:
        if str(r) not in seen and r.exists():
            out.append(r); seen.add(str(r))
    return out

def find_source_feature_matrix(outcome, aligned_df, predictor):
    """Busca una matriz de features analítica compatible con idx_original y feature_names."""
    forced = SOURCE_FEATURE_MATRIX_PATHS.get(outcome)
    if forced:
        p = Path(forced)
        if not p.exists():
            raise FileNotFoundError(f"SOURCE_FEATURE_MATRIX_PATHS[{outcome}] no existe: {p}")
        return p

    idx_col = find_col(aligned_df, ["idx_original", "filtered_idx", "row_id", "original_index", "index"], required=True, kind="row id")
    max_idx = int(np.nanmax(aligned_df[idx_col].values))
    required = predictor.feature_names if predictor.feature_names is not None else DEFAULT_FEATURES

    preferred_names = {
        "victimization": ["df_victima_feat.csv", "victimization_features.csv", "victimization_feature_matrix.csv"],
        "perpetration": ["df_perpetrador_feat.csv", "df_perpretador_feat.csv", "perpetration_features.csv", "perpetration_feature_matrix.csv"],
        "overlap": ["df_overlap_feat.csv", "overlap_features.csv", "overlap_feature_matrix.csv", "df_victima_feat.csv", "df_perpetrador_feat.csv", "lista_global_vars.csv"],
    }

    candidates = []
    for root in candidate_search_roots(BASE_DIR):
        for name in preferred_names.get(outcome, []) + ["df_victima_feat.csv", "df_perpetrador_feat.csv", "lista_global_vars.csv"]:
            candidates.extend(root.rglob(name))

    # Añade cualquier CSV con feat/features por si los nombres no coinciden.
    for root in candidate_search_roots(BASE_DIR):
        for pat in ["*feat*.csv", "*feature*.csv", "lista_global_vars.csv"]:
            candidates.extend(root.rglob(pat))

    # Dedupe
    seen, candidates2 = set(), []
    for c in candidates:
        if str(c) not in seen:
            candidates2.append(c); seen.add(str(c))

    diagnostics = []
    for p in candidates2:
        try:
            dfc = pd.read_csv(p, nrows=5)
            # necesita al menos max_idx + 1 filas: comprobación barata con lectura completa solo si parece prometedor
            full = pd.read_csv(p)
            full = ensure_alias_columns(full)
            if len(full) <= max_idx:
                diagnostics.append((str(p), full.shape, "too_few_rows"))
                continue
            _, missing = resolve_features(full.columns, configured_features=required)
            if missing:
                diagnostics.append((str(p), full.shape, f"missing {len(missing)} features"))
                continue
            print(f"{outcome}: usando matriz de features fuente: {p} | shape={full.shape}")
            return p
        except Exception as e:
            diagnostics.append((str(p), "ERROR", str(e)[:120]))

    diag_path = OUT_DIR / f"S13_source_feature_matrix_candidates_{outcome}.csv"
    pd.DataFrame(diagnostics, columns=["path", "shape", "status"]).to_csv(diag_path, index=False)
    raise ValueError(
        f"{outcome}: no encontré una matriz de features compatible. "
        f"He guardado diagnóstico en: {diag_path}\n"
        "Solución: rellena SOURCE_FEATURE_MATRIX_PATHS[outcome] con el CSV de features analítico de 3767 filas "
        "(por ejemplo df_victima_feat.csv / df_perpetrador_feat.csv) y reejecuta."
    )

def load_test_data(outcome, predictor):
    path = Path(ALIGNED_FILES[outcome])
    if not path.exists():
        raise FileNotFoundError(f"No existe aligned file para {outcome}: {path}")
    df = pd.read_csv(path)
    y_candidates = [
        f"y_true_{outcome}", f"y_{outcome}", "y_true_official", "y_true_overlap", "y_true",
        "y_test", "target", "label", "actual", "outcome"
    ]
    y_col = find_col(df, y_candidates, kind=f"y_true {outcome}")
    y = pd.to_numeric(df[y_col], errors="coerce").astype(int).values

    df_alias = ensure_alias_columns(df)
    features, missing = resolve_features(df_alias.columns, configured_features=predictor.feature_names)
    if missing or len(features) == 0:
        src_path = find_source_feature_matrix(outcome, df, predictor)
        src = pd.read_csv(src_path)
        src = ensure_alias_columns(src)
        idx_col = find_col(df, ["idx_original", "filtered_idx", "row_id", "original_index", "index"], required=True, kind="row id")
        idx = df[idx_col].values
        if SOURCE_ROW_ID_COL is None:
            Xsrc = src.iloc[idx].copy()
        else:
            Xsrc = src.set_index(SOURCE_ROW_ID_COL).loc[idx].copy()
        Xsrc = ensure_alias_columns(Xsrc)
        features, missing = resolve_features(Xsrc.columns, configured_features=predictor.feature_names)
        if missing:
            raise ValueError(f"{outcome}: siguen faltando features después de source matrix {src_path}: {missing}")
        X = Xsrc[features].copy()
    else:
        X = df_alias[features].copy()

    # Elimina columnas índice accidentales si entraran por DEFAULT_FEATURES, pero no si el modelo las espera explícitamente.
    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    if X.isna().any().any():
        bad = X.columns[X.isna().any()].tolist()
        raise ValueError(f"{outcome}: hay NaN en predictoras: {bad}")
    return X, y, features


In [60]:
from pathlib import Path

BASE_ART = Path("/Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_model_artifacts_for_S13")

# Asegurar que ARTIFACTS existe y apunta a MANUAL_ARTIFACTS
if "MANUAL_ARTIFACTS" in globals():
    ARTIFACTS = MANUAL_ARTIFACTS

# Añadir means.csv sí o sí a cada outcome
ARTIFACTS["victimization"]["means"] = str(BASE_ART / "victimization" / "means.csv")
ARTIFACTS["perpetration"]["means"] = str(BASE_ART / "perpetration" / "means.csv")
ARTIFACTS["overlap"]["means"] = str(BASE_ART / "overlap" / "means.csv")

print("ARTIFACTS means paths:")
for outcome in ["victimization", "perpetration", "overlap"]:
    p = Path(ARTIFACTS[outcome]["means"])
    print(outcome, "means =", p, "| exists:", p.exists())

ARTIFACTS means paths:
victimization means = /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_model_artifacts_for_S13/victimization/means.csv | exists: True
perpetration means = /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_model_artifacts_for_S13/perpetration/means.csv | exists: True
overlap means = /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_model_artifacts_for_S13/overlap/means.csv | exists: True


In [62]:
from pathlib import Path
import pandas as pd

for outcome in ["victimization", "perpetration", "overlap"]:
    print("\n", outcome)
    p = Path(SOURCE_FEATURE_MATRIX_PATHS[outcome])
    print("exists:", p.exists(), p)

    if p.exists():
        df_tmp = pd.read_csv(p)
        print("shape:", df_tmp.shape)
        print("columns:", df_tmp.columns.tolist())


 victimization
exists: True /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_victim/df_victima_feat.csv
shape: (3767, 28)
columns: ['Unnamed: 0', 'PAÍS', 'ETNIA.BN', 'EDAD', 'FUGAS.BN', 'ABUSOSUBS1', 'ABUSOSUBS2', 'CONVIVEN.1', 'CONVIVEN.2', 'CONVIVEN.3', 'CONVIVEN.4', 'CONVIVEN_H', 'CONVIVEN_0', 'AUTOEFIC.MEAN', 'AUTOEFIC.VAR', 'IMPULS.MEAN', 'IMPULS.MEDIAN', 'IMPULS.VAR', 'APOYO.MEAN', 'APOYO.MEDIAN', 'APOYO.VAR', 'MORAL.MEAN', 'MORAL.VAR', 'PORNO.T', 'GENERO.BN0', 'ORIENTSEX.BN0', 'GENERO.BN1', 'ORIENTSEX.BN1']

 perpetration
exists: True /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_perpetrator/content/perpetrator_v3/df_perpetrador_feat.csv
shape: (3767, 27)
columns: ['PAÍS', 'ETNIA.BN', 'EDAD', 'FUGAS.BN', 'ABUSOSUBS1', 'ABUSOSUBS2', 'CONVIVEN.1', 'CONVIVEN.2', 'CONVIVEN.3', 'CONVIVEN.4', 'CONVIVEN.5', 'CONVIVEN.6', 'AUTOEFIC.MEAN', 'AUTOEFIC.VAR', 'IMPULS.MEAN', 'IMPULS.MEDIAN', 'IMPULS.VA

In [17]:
# ======================================================================================
# VALIDACIÓN BASELINE

In [63]:
# ======================================================================================

baseline_rows = []
predictors = {}
test_data = {}

for outcome in ["victimization", "perpetration", "overlap"]:
    print("\n" + "=" * 90)
    print("Outcome:", outcome)
    predictor = FrozenPredictor(outcome, ARTIFACTS[outcome])
    X, y, features = load_test_data(outcome, predictor)
    threshold = THRESHOLDS[outcome]
    score = predictor.predict_score(X)
    baseline = compute_metrics(y, score, threshold)
    predictors[outcome] = predictor
    test_data[outcome] = (X, y, features, score, baseline)

    row = {"outcome": outcome, "n_test": len(y), "threshold": threshold, **baseline}
    problems = []
    for k, expected in OFFICIAL[outcome].items():
        observed = baseline[k]
        if abs(observed - expected) > VALIDATION_TOLERANCE:
            problems.append((k, observed, expected, observed - expected))
    row["matches_official_metrics"] = len(problems) == 0
    baseline_rows.append(row)

    print(f"n_test={len(y)} positives={int(np.sum(y))} threshold={threshold}")
    for k in ["TP", "FP", "TN", "FN", "recall_sensitivity", "specificity", "precision_ppv", "npv", "balanced_accuracy", "roc_auc", "pr_auc_average_precision"]:
        v = baseline[k]
        print(f"  {k:26s}: {v:.3f}" if isinstance(v, float) else f"  {k:26s}: {v}")

    if problems:
        print("\nBASELINE VALIDATION FAILED")
        for k, obs, exp, diff in problems:
            print(f"  {k}: observed={obs:.3f} expected={exp:.3f} diff={diff:+.3f}")
    else:
        print("Baseline validation PASSED")

baseline_df = pd.DataFrame(baseline_rows)
baseline_path = OUT_DIR / "S13_baseline_validation.csv"
baseline_df.to_csv(baseline_path, index=False)
print("\nSaved:", baseline_path)
print(baseline_df.to_string(index=False))

if not baseline_df["matches_official_metrics"].all():
    raise RuntimeError(
        "Algún baseline no reproduce métricas oficiales. NO calcules/reportes permutation importance. "
        "Revisa ARTIFACTS, aligned files y feature_names."
    )


Outcome: victimization
n_test=942 positives=465 threshold=0.5
  TP                        : 405
  FP                        : 329
  TN                        : 148
  FN                        : 60
  recall_sensitivity        : 0.871
  specificity               : 0.310
  precision_ppv             : 0.552
  npv                       : 0.712
  balanced_accuracy         : 0.591
  roc_auc                   : 0.634
  pr_auc_average_precision  : 0.588
Baseline validation PASSED

Outcome: perpetration
n_test=942 positives=221 threshold=0.5
  TP                        : 202
  FP                        : 504
  TN                        : 217
  FN                        : 19
  recall_sensitivity        : 0.914
  specificity               : 0.301
  precision_ppv             : 0.286
  npv                       : 0.919
  balanced_accuracy         : 0.607
  roc_auc                   : 0.695
  pr_auc_average_precision  : 0.398
Baseline validation PASSED

Outcome: overlap
n_test=942 positives=178 thre

In [ ]:
# ======================================================================================
# PERMUTATION IMPORTANCE

In [64]:
# ======================================================================================

all_rows = []
metrics_to_store = [
    "balanced_accuracy", "roc_auc", "pr_auc_average_precision",
    "recall_sensitivity", "specificity", "precision_ppv", "npv", "f1_positive", "accuracy"
]

for outcome in ["victimization", "perpetration", "overlap"]:
    print("\n" + "=" * 90)
    print("Permutation importance:", outcome)
    predictor = predictors[outcome]
    X, y, features, score_base, baseline = test_data[outcome]
    threshold = THRESHOLDS[outcome]
    groups = make_feature_groups(features)
    print(f"Feature groups: {len(groups)} | repeats: {N_REPEATS}")

    for gi, group in enumerate(groups, 1):
        gname = group["group"]
        members = group["features"]
        for r in range(N_REPEATS):
            seed = RANDOM_STATE + 100000 * (["victimization", "perpetration", "overlap"].index(outcome) + 1) + 1000 * gi + r
            rng = np.random.default_rng(seed)
            Xp = X.copy()
            perm_idx = rng.permutation(len(Xp))
            Xp.loc[:, members] = Xp.loc[Xp.index[perm_idx], members].to_numpy()
            score_perm = predictor.predict_score(Xp)
            m_perm = compute_metrics(y, score_perm, threshold)
            row = {
                "outcome": outcome,
                "model_name": type(predictor.model).__name__ if predictor.model is not None else type(predictor.pipeline).__name__,
                "predictor": gname,
                "predictor_domain": gname,
                "features": "|".join(members),
                "repeat": r + 1,
                "random_state": seed,
                "n_test": len(y),
                "threshold": threshold,
            }
            for m in metrics_to_store:
                row[f"baseline_{m}"] = baseline[m]
                row[f"permuted_{m}"] = m_perm[m]
                row[f"delta_{m}"] = baseline[m] - m_perm[m]
                row[f"delta_{m}_pp"] = 100 * (baseline[m] - m_perm[m])
            all_rows.append(row)
        if gi % 5 == 0 or gi == len(groups):
            print(f"  completed {gi}/{len(groups)} groups")

perm_long = pd.DataFrame(all_rows)
perm_long_path = OUT_DIR / "S13_permutation_importance_long.csv"
perm_long.to_csv(perm_long_path, index=False)
print("\nSaved:", perm_long_path)


Permutation importance: victimization
Feature groups: 25 | repeats: 30
  completed 5/25 groups
  completed 10/25 groups
  completed 15/25 groups
  completed 20/25 groups
  completed 25/25 groups

Permutation importance: perpetration
Feature groups: 27 | repeats: 30
  completed 5/27 groups
  completed 10/27 groups
  completed 15/27 groups
  completed 20/27 groups
  completed 25/27 groups
  completed 27/27 groups

Permutation importance: overlap
Feature groups: 27 | repeats: 30
  completed 5/27 groups
  completed 10/27 groups
  completed 15/27 groups
  completed 20/27 groups
  completed 25/27 groups
  completed 27/27 groups

Saved: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_permutation_importance_original_predictors_S13/S13_permutation_importance_long.csv


In [ ]:
# ======================================================================================
# SUMMARY Y TOP TABLES

In [65]:
# ======================================================================================

summary_rows = []
for (outcome, predictor, predictor_domain, features), g in perm_long.groupby(["outcome", "predictor", "predictor_domain", "features"]):
    row = {
        "outcome": outcome,
        "predictor": predictor,
        "predictor_domain": predictor_domain,
        "features": features,
        "n_repeats": len(g),
    }
    for m in metrics_to_store:
        vals = g[f"delta_{m}_pp"].astype(float).values
        row[f"{m}_drop_mean_pp"] = np.nanmean(vals)
        row[f"{m}_drop_sd_pp"] = np.nanstd(vals, ddof=1)
        row[f"{m}_drop_p2_5_pp"] = np.nanpercentile(vals, 2.5)
        row[f"{m}_drop_p97_5_pp"] = np.nanpercentile(vals, 97.5)
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary_path = OUT_DIR / "S13_permutation_importance_summary.csv"
summary.to_csv(summary_path, index=False)
print("Saved:", summary_path)

def make_top_table(summary, sort_metric="balanced_accuracy", top_n=10):
    col = f"{sort_metric}_drop_mean_pp"
    out = summary.sort_values(["outcome", col], ascending=[True, False]).groupby("outcome", as_index=False).head(top_n).copy()
    out["rank"] = out.groupby("outcome")[col].rank(method="first", ascending=False).astype(int)
    keep = [
        "outcome", "rank", "predictor", "predictor_domain", "features", "n_repeats",
        "balanced_accuracy_drop_mean_pp", "balanced_accuracy_drop_sd_pp",
        "roc_auc_drop_mean_pp", "roc_auc_drop_sd_pp",
        "pr_auc_average_precision_drop_mean_pp", "pr_auc_average_precision_drop_sd_pp",
        "specificity_drop_mean_pp", "precision_ppv_drop_mean_pp", "npv_drop_mean_pp"
    ]
    keep = [c for c in keep if c in out.columns]
    return out[keep].sort_values(["outcome", "rank"])

top_ba = make_top_table(summary, "balanced_accuracy", top_n=10)
top_pr = make_top_table(summary, "pr_auc_average_precision", top_n=10)
top_roc = make_top_table(summary, "roc_auc", top_n=10)

top_ba_path = OUT_DIR / "Table_S13_top10_by_balanced_accuracy.csv"
top_pr_path = OUT_DIR / "Table_S13_top10_by_pr_auc.csv"
top_roc_path = OUT_DIR / "Table_S13_top10_by_roc_auc.csv"

top_ba.to_csv(top_ba_path, index=False)
top_pr.to_csv(top_pr_path, index=False)
top_roc.to_csv(top_roc_path, index=False)

print("Saved:", top_ba_path)
print("Saved:", top_pr_path)
print("Saved:", top_roc_path)
print("\nTOP BA")
print(top_ba.to_string(index=False))

Saved: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_permutation_importance_original_predictors_S13/S13_permutation_importance_summary.csv
Saved: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_permutation_importance_original_predictors_S13/Table_S13_top10_by_balanced_accuracy.csv
Saved: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_permutation_importance_original_predictors_S13/Table_S13_top10_by_pr_auc.csv
Saved: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_permutation_importance_original_predictors_S13/Table_S13_top10_by_roc_auc.csv

TOP BA
      outcome  rank                         predictor                  predictor_domain                    features  n_repeats  balanced_accuracy_drop_mean_pp  balanced_accuracy_drop_sd_pp  roc_auc_drop_mean_pp  roc_auc_drop_sd_pp  pr_auc_average_precision_drop_m

In [22]:
# ======================================================================================
# FIGURAS OPCIONALES

In [66]:
# ======================================================================================

for outcome, g in summary.groupby("outcome"):
    plot_df = g.sort_values("balanced_accuracy_drop_mean_pp", ascending=False).head(15).iloc[::-1]
    fig = plt.figure(figsize=(8, 6))
    plt.barh(plot_df["predictor"], plot_df["balanced_accuracy_drop_mean_pp"])
    plt.xlabel("Mean drop in balanced accuracy (percentage points)")
    plt.ylabel("Permuted original predictor/group")
    plt.title(f"Permutation importance — {outcome}")
    plt.tight_layout()
    out_png = OUT_DIR / f"permutation_importance_{outcome}_top15_balanced_accuracy.png"
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out_png)

Saved: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_permutation_importance_original_predictors_S13/permutation_importance_overlap_top15_balanced_accuracy.png
Saved: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_permutation_importance_original_predictors_S13/permutation_importance_perpetration_top15_balanced_accuracy.png
Saved: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_permutation_importance_original_predictors_S13/permutation_importance_victimization_top15_balanced_accuracy.png


In [24]:
# ======================================================================================
# README

In [67]:
# ======================================================================================

readme_lines = [
    "S13 — Permutation importance over original engineered predictors",
    "",
    "Generated files:",
    f"- {baseline_path}",
    f"- {perm_long_path}",
    f"- {summary_path}",
    f"- {top_ba_path}",
    f"- {top_pr_path}",
    f"- {top_roc_path}",
    "",
    "Suggested supplementary table title:",
    "Supplementary Table S13. Model-Agnostic Permutation Importance of Original Engineered Predictors.",
    "",
    "Suggested note:",
    "Permutation importance was computed on the held-out internal test partition using the final frozen preprocessing and modeling pipelines. Each original engineered predictor or dummy-coded predictor group was randomly permuted while all other predictors were held fixed; the frozen scaler, PCA transformation, and classifier were then reapplied. Values represent the mean decrease in performance across repeated permutations, expressed in percentage points. Larger positive values indicate greater model-dependent performance loss after permutation. These estimates should not be interpreted as causal effects or independent predictor effects.",
    "",
    "Important validation rule:",
    "Only report these results if S13_baseline_validation.csv shows matches_official_metrics = True for all three outcomes.",
]
readme = "\n".join(readme_lines)
readme_path = OUT_DIR / "README_S13_permutation_importance.txt"
readme_path.write_text(readme, encoding="utf-8")
print("Saved:", readme_path)
print("\nDONE")

Saved: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_permutation_importance_original_predictors_S13/README_S13_permutation_importance.txt

DONE


In [68]:
from pathlib import Path

OUTDIR = Path("/Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_permutation_importance_original_predictors_S13")

for f in [
    "S13_baseline_validation.csv",
    "S13_permutation_importance_long.csv",
    "S13_permutation_importance_summary.csv",
    "Table_S13_top10_by_balanced_accuracy.csv",
    "Table_S13_top10_by_pr_auc.csv",
]:
    p = OUTDIR / f
    print(f, p.exists(), p)

S13_baseline_validation.csv True /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_permutation_importance_original_predictors_S13/S13_baseline_validation.csv
S13_permutation_importance_long.csv True /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_permutation_importance_original_predictors_S13/S13_permutation_importance_long.csv
S13_permutation_importance_summary.csv True /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_permutation_importance_original_predictors_S13/S13_permutation_importance_summary.csv
Table_S13_top10_by_balanced_accuracy.csv True /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_permutation_importance_original_predictors_S13/Table_S13_top10_by_balanced_accuracy.csv
Table_S13_top10_by_pr_auc.csv True /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_perm

In [69]:
import pandas as pd

baseline_df = pd.read_csv(OUTDIR / "S13_baseline_validation.csv")

print(baseline_df.to_string(index=False))

      outcome  n_test  threshold  TP  FP  TN  FN  recall_sensitivity  specificity  precision_ppv      npv  balanced_accuracy  f1_positive  accuracy  roc_auc  pr_auc_average_precision  matches_official_metrics
victimization     942   0.500000 405 329 148  60            0.870968     0.310273       0.551771 0.711538           0.590620     0.675563  0.587049 0.634201                  0.588061                      True
 perpetration     942   0.500000 202 504 217  19            0.914027     0.300971       0.286119 0.919492           0.607499     0.435814  0.444798 0.695069                  0.398451                      True
      overlap     942   0.523164 144 355 409  34            0.808989     0.535340       0.288577 0.923251           0.672165     0.425406  0.587049 0.753625                  0.433531                      True


In [70]:
top_ba = pd.read_csv(OUTDIR / "Table_S13_top10_by_balanced_accuracy.csv")

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", None)

cols = [
    "outcome",
    "rank",
    "predictor",
    "predictor_domain",
    "balanced_accuracy_drop_mean_pp",
    "balanced_accuracy_drop_sd_pp",
    "roc_auc_drop_mean_pp",
    "roc_auc_drop_sd_pp",
    "pr_auc_average_precision_drop_mean_pp",
    "pr_auc_average_precision_drop_sd_pp",
]

print(
    top_ba[cols]
    .sort_values(["outcome", "rank"])
    .to_string(index=False)
)

      outcome  rank                         predictor                  predictor_domain  balanced_accuracy_drop_mean_pp  balanced_accuracy_drop_sd_pp  roc_auc_drop_mean_pp  roc_auc_drop_sd_pp  pr_auc_average_precision_drop_mean_pp  pr_auc_average_precision_drop_sd_pp
      overlap     1                          FUGAS.BN                          FUGAS.BN                        1.930114                      0.773223              2.791194            0.829888                               6.249695                             1.632516
      overlap     2                           PORNO.T                           PORNO.T                        1.595290                      1.208082              3.306886            0.674351                               0.832838                             1.297444
      overlap     3                         MORAL.VAR                         MORAL.VAR                        1.312357                      0.470918              0.338206            0.256738     

In [71]:
top_pr = pd.read_csv(OUTDIR / "Table_S13_top10_by_pr_auc.csv")

cols_pr = [
    "outcome",
    "rank",
    "predictor",
    "predictor_domain",
    "pr_auc_average_precision_drop_mean_pp",
    "pr_auc_average_precision_drop_sd_pp",
    "balanced_accuracy_drop_mean_pp",
    "balanced_accuracy_drop_sd_pp",
    "roc_auc_drop_mean_pp",
    "roc_auc_drop_sd_pp",
]

print(
    top_pr[cols_pr]
    .sort_values(["outcome", "rank"])
    .to_string(index=False)
)

      outcome  rank     predictor predictor_domain  pr_auc_average_precision_drop_mean_pp  pr_auc_average_precision_drop_sd_pp  balanced_accuracy_drop_mean_pp  balanced_accuracy_drop_sd_pp  roc_auc_drop_mean_pp  roc_auc_drop_sd_pp
      overlap     1      FUGAS.BN         FUGAS.BN                               6.249695                             1.632516                        1.930114                      0.773223              2.791194            0.829888
      overlap     2    ABUSOSUBS1       ABUSOSUBS1                               2.919670                             1.148259                        0.781982                      0.773115              2.169049            0.640877
      overlap     3     APOYO.VAR        APOYO.VAR                               1.882290                             0.404219                        0.318524                      0.807960              0.912333            0.409829
      overlap     4          EDAD             EDAD                          

In [72]:
print(baseline_df.to_string(index=False))

      outcome  n_test  threshold  TP  FP  TN  FN  recall_sensitivity  specificity  precision_ppv      npv  balanced_accuracy  f1_positive  accuracy  roc_auc  pr_auc_average_precision  matches_official_metrics
victimization     942   0.500000 405 329 148  60            0.870968     0.310273       0.551771 0.711538           0.590620     0.675563  0.587049 0.634201                  0.588061                      True
 perpetration     942   0.500000 202 504 217  19            0.914027     0.300971       0.286119 0.919492           0.607499     0.435814  0.444798 0.695069                  0.398451                      True
      overlap     942   0.523164 144 355 409  34            0.808989     0.535340       0.288577 0.923251           0.672165     0.425406  0.587049 0.753625                  0.433531                      True


In [73]:
print(top_ba[cols].sort_values(["outcome", "rank"]).to_string(index=False))

      outcome  rank                         predictor                  predictor_domain  balanced_accuracy_drop_mean_pp  balanced_accuracy_drop_sd_pp  roc_auc_drop_mean_pp  roc_auc_drop_sd_pp  pr_auc_average_precision_drop_mean_pp  pr_auc_average_precision_drop_sd_pp
      overlap     1                          FUGAS.BN                          FUGAS.BN                        1.930114                      0.773223              2.791194            0.829888                               6.249695                             1.632516
      overlap     2                           PORNO.T                           PORNO.T                        1.595290                      1.208082              3.306886            0.674351                               0.832838                             1.297444
      overlap     3                         MORAL.VAR                         MORAL.VAR                        1.312357                      0.470918              0.338206            0.256738     

In [74]:
import pandas as pd
from pathlib import Path

OUTDIR = Path("/Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_permutation_importance_original_predictors_S13")

top_ba = pd.read_csv(OUTDIR / "Table_S13_top10_by_balanced_accuracy.csv")

table_s13 = top_ba.copy()

table_s13["Δ Balanced accuracy, mean (SD), pp"] = (
    table_s13["balanced_accuracy_drop_mean_pp"].round(2).astype(str)
    + " ("
    + table_s13["balanced_accuracy_drop_sd_pp"].round(2).astype(str)
    + ")"
)

table_s13["Δ ROC AUC, mean (SD), pp"] = (
    table_s13["roc_auc_drop_mean_pp"].round(2).astype(str)
    + " ("
    + table_s13["roc_auc_drop_sd_pp"].round(2).astype(str)
    + ")"
)

table_s13["Δ PR AUC, mean (SD), pp"] = (
    table_s13["pr_auc_average_precision_drop_mean_pp"].round(2).astype(str)
    + " ("
    + table_s13["pr_auc_average_precision_drop_sd_pp"].round(2).astype(str)
    + ")"
)

table_s13_final = table_s13[
    [
        "outcome",
        "rank",
        "predictor",
        "predictor_domain",
        "Δ Balanced accuracy, mean (SD), pp",
        "Δ ROC AUC, mean (SD), pp",
        "Δ PR AUC, mean (SD), pp",
    ]
].sort_values(["outcome", "rank"])

out_path = OUTDIR / "Table_S13_FINAL_for_manuscript.csv"
table_s13_final.to_csv(out_path, index=False)

print(table_s13_final.to_string(index=False))
print("\nSaved:", out_path)

      outcome  rank                         predictor                  predictor_domain Δ Balanced accuracy, mean (SD), pp Δ ROC AUC, mean (SD), pp Δ PR AUC, mean (SD), pp
      overlap     1                          FUGAS.BN                          FUGAS.BN                        1.93 (0.77)              2.79 (0.83)             6.25 (1.63)
      overlap     2                           PORNO.T                           PORNO.T                         1.6 (1.21)              3.31 (0.67)              0.83 (1.3)
      overlap     3                         MORAL.VAR                         MORAL.VAR                        1.31 (0.47)              0.34 (0.26)             0.68 (0.32)
      overlap     4                        ABUSOSUBS1                        ABUSOSUBS1                        0.78 (0.77)              2.17 (0.64)             2.92 (1.15)
      overlap     5                        CONVIVEN.2                        CONVIVEN.2                        0.77 (0.69)              0.71

## Qué enviar después

Cuando termine, pásame estos archivos o sus tablas impresas:

- `S13_baseline_validation.csv`
- `Table_S13_top10_by_balanced_accuracy.csv`
- `Table_S13_top10_by_pr_auc.csv`
- `S13_permutation_importance_summary.csv`

Si se para por artefactos no encontrados, pásame `S13_artifact_candidates.csv` y el bloque impreso de `ARTIFACTS`.


In [20]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

BASE = Path("/Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final")

aligned_path = BASE / "aligned_test_exports/overlap_aligned_test_rowlevel.csv"
aligned = pd.read_csv(aligned_path)

print("aligned overlap:", aligned.shape)
print(aligned.columns.tolist())

# Detectar columnas
y_col = "y_true_official" if "y_true_official" in aligned.columns else "y_true"
pred_col = "y_pred" if "y_pred" in aligned.columns else None
prob_col = "y_prob" if "y_prob" in aligned.columns else None

print("y_col:", y_col)
print("pred_col:", pred_col)
print("prob_col:", prob_col)

y = aligned[y_col].astype(int).values

def print_metrics(name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    recall = tp / (tp + fn)
    spec = tn / (tn + fp)
    ppv = tp / (tp + fp)
    npv = tn / (tn + fn)
    ba = (recall + spec) / 2
    acc = (tp + tn) / (tp + tn + fp + fn)

    print("\n", name)
    print("TP FP TN FN:", tp, fp, tn, fn)
    print("recall:", round(recall, 6))
    print("specificity:", round(spec, 6))
    print("ppv:", round(ppv, 6))
    print("npv:", round(npv, 6))
    print("balanced_accuracy:", round(ba, 6))
    print("accuracy:", round(acc, 6))

# 1. Métricas oficiales guardadas en aligned
if pred_col is not None:
    y_pred_official = aligned[pred_col].astype(int).values
    print_metrics("OFFICIAL aligned y_pred", y, y_pred_official)

# 2. Métricas con y_prob oficial a threshold 0.5
if prob_col is not None:
    y_prob_official = aligned[prob_col].astype(float).values
    y_pred_prob05 = (y_prob_official >= 0.5).astype(int)
    print_metrics("OFFICIAL aligned y_prob >= 0.5", y, y_pred_prob05)

# 3. Métricas del modelo S13 actual
score_model = test_data["overlap"][3]
y_pred_model05 = (score_model >= 0.5).astype(int)
print_metrics("S13 exported model score >= 0.5", y, y_pred_model05)

# 4. Comparar probabilidades oficiales vs modelo exportado
if prob_col is not None:
    diff = score_model - y_prob_official
    print("\nScore comparison: S13 model vs aligned y_prob")
    print("mean diff:", float(np.mean(diff)))
    print("max abs diff:", float(np.max(np.abs(diff))))
    print("corr:", float(np.corrcoef(score_model, y_prob_official)[0, 1]))
    print("first 10 diffs:", diff[:10])

# 5. Buscar threshold del modelo S13 que reproduce counts oficiales Table 1
expected_counts = {
    "TP": 144,
    "FP": 355,
    "TN": 409,
    "FN": 34,
}

rows = []
for thr in np.linspace(0.01, 0.99, 991):
    pred = (score_model >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    rows.append({
        "threshold": thr,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
        "diff_counts": abs(tp - expected_counts["TP"]) + abs(fp - expected_counts["FP"]) + abs(tn - expected_counts["TN"]) + abs(fn - expected_counts["FN"]),
    })

thr_df = pd.DataFrame(rows).sort_values("diff_counts")
print("\nBest thresholds for S13 model to match expected Table 1 counts:")
print(thr_df.head(20).to_string(index=False))

# 6. Buscar threshold del y_prob oficial que reproduce counts oficiales Table 1
if prob_col is not None:
    rows2 = []
    for thr in np.linspace(0.01, 0.99, 991):
        pred = (y_prob_official >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
        rows2.append({
            "threshold": thr,
            "TP": tp,
            "FP": fp,
            "TN": tn,
            "FN": fn,
            "diff_counts": abs(tp - expected_counts["TP"]) + abs(fp - expected_counts["FP"]) + abs(tn - expected_counts["TN"]) + abs(fn - expected_counts["FN"]),
        })

    thr_df2 = pd.DataFrame(rows2).sort_values("diff_counts")
    print("\nBest thresholds for aligned y_prob to match expected Table 1 counts:")
    print(thr_df2.head(20).to_string(index=False))

aligned overlap: (942, 26)
['idx_original', 'y_true', 'y_pred', 'y_prob_pos', 'y_true_overlap', 'y_pred_overlap', 'y_prob_overlap', 'V.SUM.TOTAL', 'P.SUM.TOTAL', 'PAÍS', 'ETNIA.BN', 'EDAD', 'GENERO_BIN_0', 'GENERO_BIN_1', 'ORIENTSEX.BN_1', 'ORIENTSEX.BN_2', 'INTERSECT', 'victim_count_ge1', 'victim_count_ge2', 'victim_count_ge3', 'perp_count_ge1', 'perp_count_ge2', 'perp_count_ge3', 'overlap_count_ge1', 'overlap_count_ge2', 'overlap_count_ge3']
y_col: y_true
pred_col: y_pred
prob_col: None

 OFFICIAL aligned y_pred
TP FP TN FN: 144 355 409 34
recall: 0.808989
specificity: 0.53534
ppv: 0.288577
npv: 0.923251
balanced_accuracy: 0.672165
accuracy: 0.587049

 S13 exported model score >= 0.5
TP FP TN FN: 153 407 357 25
recall: 0.859551
specificity: 0.467277
ppv: 0.273214
npv: 0.934555
balanced_accuracy: 0.663414
accuracy: 0.541401

Best thresholds for S13 model to match expected Table 1 counts:
 threshold  TP  FP  TN  FN  diff_counts
  0.536626 145 355 409  33            2
  0.537616 145 354

In [26]:
baseline_df[["outcome", "TP", "FP", "TN", "FN", "recall_sensitivity", "specificity", "balanced_accuracy", "matches_official_metrics"]]

,outcome,TP,FP,TN,FN,recall_sensitivity,specificity,balanced_accuracy,matches_official_metrics
0,victimization,405,329,148,60,0.870968,0.310273,0.590620,True
1,perpetration,202,504,217,19,0.914027,0.300971,0.607499,True
2,overlap,153,407,357,25,0.859551,0.467277,0.663414,False


In [27]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

y = test_data["overlap"][1]
score = test_data["overlap"][3]

expected = {"TP": 144, "FP": 355, "TN": 409, "FN": 34}

rows = []
for thr in np.linspace(0.01, 0.99, 9901):
    pred = (score >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()

    rows.append({
        "threshold": thr,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
        "diff": abs(tp - expected["TP"]) + abs(fp - expected["FP"]) + abs(tn - expected["TN"]) + abs(fn - expected["FN"]),
        "recall": tp / (tp + fn),
        "specificity": tn / (tn + fp),
        "balanced_accuracy": ((tp / (tp + fn)) + (tn / (tn + fp))) / 2,
    })

thr_df = pd.DataFrame(rows).sort_values("diff")
print(thr_df.head(30).to_string(index=False))

 threshold  TP  FP  TN  FN  diff   recall  specificity  balanced_accuracy
  0.536626 145 355 409  33     2 0.814607     0.535340           0.674974
  0.535834 146 355 409  32     4 0.820225     0.535340           0.677783
  0.537418 145 354 410  33     4 0.814607     0.536649           0.675628
  0.537319 145 354 410  33     4 0.814607     0.536649           0.675628
  0.537220 145 354 410  33     4 0.814607     0.536649           0.675628
  0.537121 145 354 410  33     4 0.814607     0.536649           0.675628
  0.537022 145 354 410  33     4 0.814607     0.536649           0.675628
  0.536923 145 354 410  33     4 0.814607     0.536649           0.675628
  0.536824 145 354 410  33     4 0.814607     0.536649           0.675628
  0.536725 145 354 410  33     4 0.814607     0.536649           0.675628
  0.537616 145 354 410  33     4 0.814607     0.536649           0.675628
  0.536527 146 355 409  32     4 0.820225     0.535340           0.677783
  0.536329 146 355 409  32     4 0.820

In [28]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

y = test_data["overlap"][1]
score = test_data["overlap"][3]

expected = {"TP": 144, "FP": 355, "TN": 409, "FN": 34}

rows = []

# candidatos: todos los scores únicos y puntos medios entre scores
unique_scores = np.sort(np.unique(score))
midpoints = (unique_scores[:-1] + unique_scores[1:]) / 2
thresholds = np.unique(np.concatenate([unique_scores, midpoints, np.linspace(0.01, 0.99, 9901)]))

for op in [">=", ">"]:
    for thr in thresholds:
        if op == ">=":
            pred = (score >= thr).astype(int)
        else:
            pred = (score > thr).astype(int)

        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()

        rows.append({
            "operator": op,
            "threshold": thr,
            "TP": tp,
            "FP": fp,
            "TN": tn,
            "FN": fn,
            "diff": abs(tp - expected["TP"]) + abs(fp - expected["FP"]) + abs(tn - expected["TN"]) + abs(fn - expected["FN"]),
            "recall": tp / (tp + fn),
            "specificity": tn / (tn + fp),
            "balanced_accuracy": ((tp / (tp + fn)) + (tn / (tn + fp))) / 2,
        })

thr_exact = pd.DataFrame(rows).sort_values(["diff", "threshold"])

print(thr_exact.head(50).to_string(index=False))

exact = thr_exact[thr_exact["diff"] == 0]
print("\nEXACT MATCHES")
print(exact.to_string(index=False))

operator  threshold  TP  FP  TN  FN  diff   recall  specificity  balanced_accuracy
       >   0.536621 145 355 409  33     2 0.814607     0.535340           0.674974
      >=   0.536626 145 355 409  33     2 0.814607     0.535340           0.674974
       >   0.536626 145 355 409  33     2 0.814607     0.535340           0.674974
      >=   0.536660 145 355 409  33     2 0.814607     0.535340           0.674974
       >   0.536660 145 355 409  33     2 0.814607     0.535340           0.674974
      >=   0.536699 145 355 409  33     2 0.814607     0.535340           0.674974
       >   0.535350 146 355 409  32     4 0.820225     0.535340           0.677783
      >=   0.535438 146 355 409  32     4 0.820225     0.535340           0.677783
       >   0.535438 146 355 409  32     4 0.820225     0.535340           0.677783
      >=   0.535537 146 355 409  32     4 0.820225     0.535340           0.677783
       >   0.535537 146 355 409  32     4 0.820225     0.535340           0.677783
    